# 34MLS — Assignment 3: Flow Field Through Porous Media

**Student name:** [Your name here]  
**Student ID:** [Your ID here]  
**Kaggle ID:** [Your Kaggle username here]  
**Deadline:** 11-04-2026 23:59

---

## About this notebook

This notebook is the appendix to the report `assignment3.tex`. It contains the full implementation code, structured in the same order as the report sections.

**Rubric map:**
- Step 1 (this section): Data exploration → feeds report **Section A** (dataset characterization)
- Steps 2–4: Preprocessing, augmentation → feeds report **Section A** (loss functions) and **Section B**
- Steps 5–12: Model implementation, training, comparisons → feeds report **Section C**
- Step 13: Kaggle submission → **Section D**

## Reproducibility seeds

We set seeds at the very top so that every random operation in this notebook — data splits, weight initialization, augmentation — produces the same result every time. This is required by **rubric E** (a peer must be able to reproduce our results).

- `torch.manual_seed` controls PyTorch random operations (weight init, dropout)
- `np.random.seed` controls NumPy random operations (shuffling, splits)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import os, json, time

# Reproducibility — must be set before any random operation
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device — use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Paths
DATA_DIR = 'flow-field-through-a-porous-media-25-26'
os.makedirs('experiments/logs',        exist_ok=True)
os.makedirs('experiments/plots',       exist_ok=True)
os.makedirs('experiments/checkpoints', exist_ok=True)
os.makedirs('submissions',             exist_ok=True)
os.makedirs('fig',                     exist_ok=True)

---
## Step 1 — Data Exploration

**What this step does:** Load all three training files and visualise the data before touching any model.

**Why before anything else?** Report section A requires us to *characterise the dataset in statistical terms* (rubric A, sub-point 3). We cannot write about symmetries or dimensional analysis without first having *seen* what the cross-sections and flow fields look like. This step produces the figures and numbers that will go directly into the report.

**The three files we load:**

| File | What it contains | Shape |
|------|-----------------|-------|
| `train_inputs.npy` | Binary cross-sections: 1 = open pixel, 0 = closed pixel | [1500, 32, 64] |
| `train_params.csv` | Physical parameters per sample: ΔP, L, μ, ΔA | [1500, 5 cols] |
| `train_labels.npy` | Ground truth flow rate field q(x,y) [m/s] | [1500, 32, 64] |

In [ ]:
# Load all training data
train_inputs = np.load(f'{DATA_DIR}/train_inputs.npy')      # binary cross-sections
train_labels = np.load(f'{DATA_DIR}/train_labels.npy')      # flow rate fields q(x,y)
train_params = pd.read_csv(f'{DATA_DIR}/train_params.csv')  # physical parameters

print("=== Shapes ===")
print(f"train_inputs : {train_inputs.shape}  dtype={train_inputs.dtype}")
print(f"train_labels : {train_labels.shape}  dtype={train_labels.dtype}")
print(f"train_params : {train_params.shape}")
print()
print("=== Parameter columns ===")
print(train_params.head())
print()
print("=== Input pixel values (should be 0 and 1 only) ===")
print(f"Unique values in inputs: {np.unique(train_inputs)}")
print()
print("=== Flow field range ===")
print(f"q min:  {train_labels.min():.4e} m/s")
print(f"q max:  {train_labels.max():.4e} m/s")
print(f"q mean: {train_labels.mean():.4e} m/s")

### 1.1 — Example cross-sections and flow fields

**What we plot:** 6 randomly chosen samples. For each: the binary cross-section (left) and the corresponding flow field q(x,y) (right).

**What to look for:**
- The flow field is **zero wherever the pixel is closed (=0)**. You cannot have flow through a wall — this is a hard physical constraint our model must respect.
- The flow field is **symmetric in a specific way**: if the cross-section has a left-right symmetry, so does q(x,y). This is the equivariance we will discuss in report section A.
- The magnitude of q varies across the open region — pixels near the centre of an open channel flow faster than pixels near the edges (boundary layer effect).

In [ ]:
np.random.seed(SEED)  # ensure same samples every run
sample_ids = np.random.choice(len(train_inputs), size=6, replace=False)

fig, axes = plt.subplots(6, 2, figsize=(10, 18))
fig.suptitle('Example cross-sections and flow fields', fontsize=14, y=1.01)

for row, idx in enumerate(sample_ids):
    # Cross-section (binary image)
    ax_cs = axes[row, 0]
    im0 = ax_cs.imshow(train_inputs[idx], cmap='gray', vmin=0, vmax=1, aspect='auto')
    ax_cs.set_title(f'Sample {idx} — Cross-section')
    ax_cs.set_xlabel('y pixel index')
    ax_cs.set_ylabel('x pixel index')
    plt.colorbar(im0, ax=ax_cs, label='open (1) / closed (0)')

    # Flow field
    ax_fl = axes[row, 1]
    im1 = ax_fl.imshow(train_labels[idx], cmap='viridis', aspect='auto')
    ax_fl.set_title(f'Sample {idx} — Flow field q(x,y) [m/s]')
    ax_fl.set_xlabel('y pixel index')
    ax_fl.set_ylabel('x pixel index')
    plt.colorbar(im1, ax=ax_fl, label='q [m/s]')

plt.tight_layout()
plt.savefig('experiments/plots/exploration_cross_sections_and_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_cross_sections_and_flow.png")

### 1.2 — Physical parameter distributions

**What we plot:** Histograms of the four physical parameters across all 1500 training samples.

**Why this matters for the report:**
- Report section A asks us to *characterise the dataset in statistical terms*.
- The parameter distributions tell us how diverse the training set is — are we training on a narrow range of viscosities, or a wide one?
- The distributions also inform **dimensional analysis**: if ΔP, L, μ, ΔA all vary independently, we must make our model robust to all combinations. A model that hard-codes a specific viscosity will fail on the test set.

**Physical reminder of what each parameter means:**
- **ΔP [Pa]** — pressure drop driving the flow. Higher ΔP → faster flow. q scales linearly with ΔP (Darcy's law).
- **L [m]** — channel length. Longer channel → slower flow (more resistance). q scales as 1/L.
- **μ [Pa·s]** — fluid viscosity. More viscous fluid → slower flow. q scales as 1/μ.
- **ΔA [m²]** — pixel area. Affects the physical size of each open region.

In [ ]:
param_cols = ['delta_p', 'L', 'visc', 'delta_A']
param_labels = [
    r'$\Delta P$ [Pa]',
    r'$L$ [m]',
    r'$\mu$ [Pa$\cdot$s]',
    r'$\Delta A$ [m$^2$]'
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Distribution of physical parameters across 1500 training samples', fontsize=13)

for ax, col, label in zip(axes.flat, param_cols, param_labels):
    values = train_params[col].values
    ax.hist(values, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Count')
    ax.set_title(f'{label}  |  min={values.min():.3e}  max={values.max():.3e}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/exploration_parameter_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_parameter_distributions.png")

print("\n=== Parameter summary statistics ===")
print(train_params[param_cols].describe().to_string())

### 1.3 — Flow field value distribution

**What we plot:**
1. Histogram of ALL q values across all 1500 samples (one value per pixel per sample = 1500 × 32 × 64 = 3,072,000 values).
2. Distribution of the **total volumetric flow** Q per sample, where Q = Σ q(x,y) · ΔA [m³/s].

**Why:**
- The q histogram will show a large spike at zero (all closed pixels have q=0). The non-zero values tell us the actual flow rate magnitudes we need to predict.
- Q per sample tells us how much the total flow varies across the dataset — this matters for choosing a good loss function.
- These plots go directly into the report (rubric A, dataset characterisation).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Flow field statistics across training set', fontsize=13)

# --- Plot 1: all q values (log scale to show the zero spike) ---
ax = axes[0]
all_q = train_labels.flatten()
ax.hist(all_q, bins=80, color='steelblue', edgecolor='none', log=True)
ax.set_xlabel('q(x,y) [m/s]')
ax.set_ylabel('Count (log scale)')
ax.set_title('All q values\n(log-scale count)')
ax.grid(True, alpha=0.3)

# --- Plot 2: non-zero q values only ---
ax = axes[1]
nonzero_q = all_q[all_q > 0]
ax.hist(nonzero_q, bins=80, color='darkorange', edgecolor='none')
ax.set_xlabel('q(x,y) [m/s]')
ax.set_ylabel('Count')
ax.set_title(f'Non-zero q values only\n({len(nonzero_q):,} pixels open)')
ax.grid(True, alpha=0.3)

# --- Plot 3: total volumetric flow Q per sample ---
ax = axes[2]
# Q = sum of q(x,y) * delta_A for each sample
# delta_A varies per sample, so we use the per-sample value from params
delta_A_vals = train_params['delta_A'].values          # shape [1500]
q_sums = train_labels.reshape(len(train_labels), -1).sum(axis=1)  # sum over pixels
Q_per_sample = q_sums * delta_A_vals                   # Q [m³/s] per sample
ax.hist(Q_per_sample, bins=60, color='seagreen', edgecolor='none')
ax.set_xlabel(r'$Q = \sum_{xy} q(x,y) \cdot \Delta A$ [m³/s]')
ax.set_ylabel('Count')
ax.set_title('Total volumetric flow Q per sample')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/exploration_flow_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_flow_distribution.png")

print(f"\nNon-zero q pixels: {len(nonzero_q):,} / {len(all_q):,} total ({100*len(nonzero_q)/len(all_q):.1f}% open)")
print(f"Q range: [{Q_per_sample.min():.3e}, {Q_per_sample.max():.3e}] m³/s")
print(f"Q mean:   {Q_per_sample.mean():.3e} m³/s")

### 1.4 — Open pixel fraction per sample

**What we plot:** For each of the 1500 samples, what fraction of the 32×64 = 2048 pixels are open (=1)?

**Why this matters:**
- This tells us how porous the media is on average. A sample with 10% open pixels is very dense (little flow); one with 80% is nearly empty (high flow).
- The distribution of open fractions tells us whether the training set is balanced or biased toward one type of medium.
- It also gives us intuition for the *non-physical output* problem (rubric B): a network predicting q > 0 on closed pixels is physically wrong, regardless of how small the value is.

In [ ]:
# Open fraction = number of pixels with value 1 / total pixels per sample
open_fractions = train_inputs.reshape(len(train_inputs), -1).mean(axis=1)  # shape [1500]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Open pixel fraction across training samples', fontsize=13)

# Histogram
ax = axes[0]
ax.hist(open_fractions, bins=50, color='mediumpurple', edgecolor='none')
ax.set_xlabel('Open pixel fraction (porosity)')
ax.set_ylabel('Number of samples')
ax.set_title('Distribution of porosity')
ax.axvline(open_fractions.mean(), color='red', linestyle='--', label=f'Mean = {open_fractions.mean():.2f}')
ax.legend()
ax.grid(True, alpha=0.3)

# Scatter: porosity vs total flow Q
ax = axes[1]
ax.scatter(open_fractions, Q_per_sample, alpha=0.3, s=8, color='steelblue')
ax.set_xlabel('Open pixel fraction (porosity)')
ax.set_ylabel(r'Total flow $Q$ [m³/s]')
ax.set_title('Porosity vs total volumetric flow')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/exploration_porosity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_porosity.png")

print(f"\nOpen fraction — min: {open_fractions.min():.2f}  max: {open_fractions.max():.2f}  mean: {open_fractions.mean():.2f}")

### 1.5 — Exploration summary

What we now know from the data, and how it feeds into the report:

| Observation | Report implication |
|-------------|-------------------|
| q = 0 exactly on closed pixels | Hard physical constraint → non-physical output prevention (rubric B) |
| q scales with ΔP, 1/L, 1/μ | Dimensional analysis → degree-1 homogeneous form (rubric A) |
| Cross-sections have left-right and up-down symmetry potential | Symmetry analysis → equivariance of q(x,y) (rubric A) |
| Parameters vary independently across samples | Model must generalise across all (ΔP, L, μ, ΔA) combinations |
| Porosity varies widely | Loss function must handle both near-zero and large flow magnitudes robustly |

**Next step:** Preprocessing pipeline — normalisation, train/validation split, PyTorch DataLoaders.